In [422]:
import re
import sys
from pathlib import Path

import torch
import torch.nn.functional as F

from models.lstm_model import LSTM_Model
from utils import get_device,tokenize,encode_text,NLP_data_cleaning


# Parameters

In [423]:
candidate_checkpoint_paths = [
    Path("checkpoints/sentiment_checkpoint.pt"),
    Path("../checkpoints/sentiment_checkpoint.pt"),
    Path("../../checkpoints/sentiment_checkpoint.pt"),
    Path("../../../checkpoints/sentiment_checkpoint.pt"),
]

checkpoint_path = None
for p in candidate_checkpoint_paths:
    p_abs = p.resolve()
    if p_abs.exists():
        checkpoint_path = str(p_abs)
        break

if checkpoint_path is None:
    raise FileNotFoundError(
        "Could not find sentiment checkpoint. Tried: "
        + ", ".join(str(p.resolve()) for p in candidate_checkpoint_paths)
    )

print(f"Using checkpoint: {checkpoint_path}")

Using checkpoint: C:\Users\ada_lau\Documents\Work\Learn\pytorch-usages\checkpoints\sentiment_checkpoint.pt


## Extract saved Objects and config

In [424]:
device = get_device()

checkpoint = torch.load(checkpoint_path, map_location=device)

In [425]:
model_config = checkpoint["model_config"]
vocab = checkpoint["vocab"]
label_map = checkpoint["label_map"]

In [426]:
preprocess_config = checkpoint.get("preprocess_config", {})
max_len = int(preprocess_config.get("seq_length", checkpoint.get("max_len", 40)))
pad_idx = int(preprocess_config.get("padding_idx", 0))
unk_idx = int(preprocess_config.get("unk_idx", 1))

In [427]:
id_to_label = {int(v): k for k, v in label_map.items()}

## Rebuild model

In [428]:
model = LSTM_Model(**model_config).to(device)
model.load_state_dict(checkpoint["model_state_dict"])
_= model.eval()

# Functions

## Function to preprocess data

In [429]:
def data_preprocess( texts ):

    texts = [ NLP_data_cleaning(text) for text in texts]
    
    X = [
        encode_text(text=t, vocab=vocab, unk_idx=unk_idx, pad_idx=pad_idx, max_len=max_len)
        for t in texts
    ]

    X_tensor = torch.tensor(X, dtype=torch.long, device=device)

    return X_tensor
    

## Function to predict

In [430]:
def predict_sentiment( texts, X_tensor ):

    logits = model(X_tensor)
    probs = F.softmax(logits, dim=1)  
    
    pred_ids = probs.argmax(dim=1).cpu().tolist()
    pred_confs = probs.max(dim=1).values.cpu().tolist()
    
    results = []
    
    for text, cls_id, conf, p in zip(texts, pred_ids, pred_confs, probs.cpu().tolist()):
        
        label = id_to_label.get(cls_id, str(cls_id))
        original_pred_label = ''

        if conf < 0.5 and label != 'neutral':

            pred_ids = label_map.get('neutral')
            original_pred_label = label
            label = 'neutral'          
        
            
        results.append({
                "pred_id": cls_id,
                "pred_label": label,
                "original_pred_label" : original_pred_label,
                "confidence": float(conf),
                "probs": [float(x) for x in p],     
                "text": text,
            })    


    return results

## Main function to process and predict sentiment of the input list of sentense

In [431]:
def main_sentiment_predict(texts, actual = None):
    
    correct = 0
    incorrect = 0

    processed_X = data_preprocess( texts )
    
    result = predict_sentiment( texts , processed_X )

    
    for item in result:

        text = item['text']
        predict = item['pred_label']
        confid = item['confidence']

        
        if actual is not None:
            
            if actual == predict:
                compare = 'Matched !!'
                correct += 1
            else:
                compare = 'Wrong !!'
                incorrect+= 1         

        else:

            actual = '--'
            compare = ''

            
        if compare != 'Matched !!':
            
            result_item = item.copy()
            result_item = {'Actual': actual, **{k: v for k, v in result_item.items() if k != 'Acutal'}}

            
            print(f"{compare}")      
            
            for k,v in result_item.items():

                if k not in ['pred_id', 'probs'] and v != '':
                   print(f"{k.ljust(20, ' ')} : {v}")

            print('\n')

    return correct,incorrect
    

# Get Data

In [432]:
positive_texts = [
    "Company reported strong earnings and raised guidance.",
    "Apple beats Q3 estimates with record iPhone sales.",
    "Tesla shares surge after strong delivery numbers.",
    "Fed signals pause in rate hikes, markets rally.",
    "Amazon announces $10 billion stock buyback program.",
    "Nvidia profit triples on surging AI chip demand.",
    "Goldman Sachs upgrades stock to buy with $200 target.",
    "Microsoft posts better-than-expected cloud revenue and lifts outlook.",
    "Netflix adds more subscribers than forecast and shares jump.",
    "Company secures a major contract that boosts future revenue visibility.",
    "Strong quarterly margins drive upbeat guidance for the rest of the year.",
    "Analysts raise earnings estimates after another solid quarter.",
    "The merger is expected to create significant cost savings and synergies.",
    "Robust consumer demand helped the retailer deliver record holiday sales.",
    "The pharmaceutical company won regulatory approval for its new blockbuster drug.",
    "Lower input costs and higher prices improved profit across all segments.",
]

negative_texts = [
    "Revenue missed expectations and outlook remains weak.",
    "Company slashes full-year forecast amid supply chain issues.",
    "Bank reports massive write-down on bad loans.",
    "Layoffs announced as tech firm struggles with slowing growth.",
    "Regulators probe firm over accounting irregularities.",
    "Oil prices plunge on recession fears and weak demand.",
    "Credit rating downgraded to junk status by Moody's.",
    "The company posted a wider-than-expected loss and suspended guidance.",
    "Shares fell after management warned of declining demand next quarter.",
    "Rising costs and weak sales forced the manufacturer to cut jobs.",
    "A product recall is expected to hurt earnings and damage the brand.",
    "The airline reported soft bookings and a sharp drop in operating profit.",
    "Default risk increased after the firm missed a major debt payment.",
    "Analysts downgraded the stock after disappointing results and weak margins.",
    "Export restrictions are likely to reduce semiconductor revenue this year.",
    "The retailer closed dozens of stores after another quarter of falling sales.",
]

neutral_texts = [
    "Stock traded flat after mixed quarterly results.",
    "Board meeting scheduled for next Tuesday.",
    "Company files annual report with the SEC.",
    "CEO to present at investor conference next month.",
    "Merger expected to close in Q4 pending regulatory approval.",
    "Analyst maintains hold rating with unchanged price target.",
    "Trading volume was below average in afternoon session.",
    "The company announced the date of its next earnings release.",
    "Management will host a conference call after the market closes.",
    "Shares were little changed in premarket trading.",
    "The firm renewed its existing credit facility with the same terms.",
    "Directors approved the regular quarterly dividend payment.",
    "A routine tax filing was submitted to the exchange this morning.",
    "The manufacturer opened a new regional office in Singapore.",
    "The bank completed its planned leadership transition this week.",
    "The company confirmed that previously announced guidance remains unchanged.",
]

# Predict

In [433]:
correct_pos,incorrect_pos = main_sentiment_predict(positive_texts , 'positive')
correct_neg,incorrect_neg = main_sentiment_predict(negative_texts , 'negative')
correct_neu,incorrect_neu = main_sentiment_predict(neutral_texts , 'neutral')

Wrong !!
Actual               : positive
pred_label           : neutral
original_pred_label  : negative
confidence           : 0.4339386224746704
text                 : Fed signals pause in rate hikes, markets rally.


Wrong !!
Actual               : positive
pred_label           : neutral
original_pred_label  : positive
confidence           : 0.484266072511673
text                 : Nvidia profit triples on surging AI chip demand.


Wrong !!
Actual               : positive
pred_label           : neutral
confidence           : 0.6745072603225708
text                 : The merger is expected to create significant cost savings and synergies.


Wrong !!
Actual               : positive
pred_label           : neutral
original_pred_label  : negative
confidence           : 0.4457629919052124
text                 : Lower input costs and higher prices improved profit across all segments.


Wrong !!
Actual               : negative
pred_label           : neutral
original_pred_label  : negative
co

In [434]:
print(f'for Positive: Correct - {correct_pos} and Incorrect = {incorrect_pos}')
print(f'for Negative: Correct - {correct_neg} and Incorrect = {incorrect_neg}')
print(f'for Neutral : Correct - {correct_neu} and Incorrect = {incorrect_neu}')

for Positive: Correct - 12 and Incorrect = 4
for Negative: Correct - 13 and Incorrect = 3
for Neutral: Correct - 14 and Incorrect = 2


# Test

In [435]:
# for item in negative_texts:

#     processed = NLP_data_cleaning(item)

#     print(item)
#     print(processed)
#     print('\n')